# 02 — Background removal (GPU-accelerated)

Ports `background_remover.py`. The original uses `rembg`'s default CPU
session. Here the `rembg` session is built with `onnxruntime-gpu` and
`CUDAExecutionProvider`, which is meaningfully faster once you're removing
backgrounds from hundreds of scraped photos per snack.

Run `00_environment_setup.ipynb` first — this notebook assumes
`CUDAExecutionProvider` is available.


In [ ]:
import os
from rembg import remove, new_session
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt


## GPU session

`rembg.new_session` accepts an ONNX Runtime provider list. If
`CUDAExecutionProvider` fails to initialize, `onnxruntime` silently falls back
to CPU — the check below makes that failure loud instead of silent.


In [ ]:
MODEL_NAME = "u2net"  # rembg's default general-purpose model

session = new_session(
    model_name=MODEL_NAME,
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)

active_providers = session.inner_session.get_providers()
print("active providers (in priority order):", active_providers)
assert active_providers[0] == "CUDAExecutionProvider", (
    "rembg fell back to CPU. Re-check onnxruntime-gpu install / CUDA version match."
)


## Config

In [ ]:
INPUT_FOLDER = "nutri_grain_strawberry_bar"
OUTPUT_FOLDER = f"{INPUT_FOLDER}_transparent"


## Intermediary test — process one image and inspect the cutout

Displays the original next to the result composited over a checkerboard, so a
bad/incomplete cutout (common failure: matte packaging confusing the matting
model) is obvious before you burn GPU time on the whole folder.


In [ ]:
def checkerboard(size, tile=16):
    w, h = size
    board = np.zeros((h, w, 3), dtype=np.uint8)
    for y in range(0, h, tile):
        for x in range(0, w, tile):
            if ((x // tile) + (y // tile)) % 2 == 0:
                board[y:y + tile, x:x + tile] = 200
            else:
                board[y:y + tile, x:x + tile] = 255
    return Image.fromarray(board)


def remove_bg(input_path):
    input_img = Image.open(input_path)
    return remove(input_img, session=session)


sample_files = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))]
assert sample_files, f"No images found in '{INPUT_FOLDER}' — run 01_data_scraper.ipynb first."

sample_path = os.path.join(INPUT_FOLDER, sample_files[0])
sample_out = remove_bg(sample_path)

bg_check = checkerboard(sample_out.size)
bg_check.paste(sample_out, (0, 0), sample_out)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(Image.open(sample_path))
axes[0].set_title("original")
axes[1].imshow(bg_check)
axes[1].set_title("cutout over checkerboard")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Batch processing

In [ ]:
def automate_local_remove_bg(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if not filename.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
            continue

        input_path = os.path.join(input_folder, filename)
        base_name = os.path.splitext(filename)[0]
        output_path = os.path.join(output_folder, f"{base_name}_transparent.png")

        print(f"Processing '{filename}'...")
        try:
            output_img = remove_bg(input_path)
            output_img.save(output_path)
            print(f"  -> saved {output_path}")
        except Exception as e:
            print(f"  -> failed on {filename}: {e}")

    print("\nDone! All backgrounds removed on GPU.")


In [ ]:
RUN_FULL_BATCH = False  # flip to True once the single-image test above looks right

if RUN_FULL_BATCH:
    automate_local_remove_bg(INPUT_FOLDER, OUTPUT_FOLDER)
